# 🧪 実験: 既存BitNetモデルのFFN(MLP)重み移植は学習を速くするか？

**仮説**: SSM部分(系列混合)はゼロから学習するしかないが、FFN(SwiGLU)は「トークンごとに独立したゲート付きMLP」という
インターフェースがTransformerでもSSMブロックでも同じなので、既存の学習済みBitNet系モデルのFFN重みを凍結して移植すれば、
SSM側がそれに合う入力分布を学習することで、完全ランダム初期化より速く収束するのではないか？

**先行研究の根拠**: ["Pretrained Transformers as Universal Computation Engines" (Lu et al., 2021)](https://arxiv.org/abs/2103.05247) は、
事前学習済みTransformerブロックを凍結したまま全く異なるタスクに挿入しても、周囲だけ学習させれば機能することを示した。
ただしこれは「Attention+FFNをブロックごと・float精度で」凍結した結果であり、今回は「FFN単体・三値量子化」という
より条件の厳しいケースなので、**効果があるかは未検証**。だからこそ実験する。

**ドナーモデル**: [`abideen/Bitnet-Llama-70M`](https://huggingface.co/abideen/Bitnet-Llama-70M) — 
`hidden_size=768`(このリポジトリの"Small"ティアと一致)、`intermediate_size=1024`、6層。
FFNの次元をこれに合わせて構成すれば、パディングや変換層なしでそのまま重みをコピーできる。

**比較する2モデル**(アーキテクチャは完全に同一、違いはFFN初期化のみ):
- **Variant A (transplant)**: 6層すべてのFFN(`ffn_in`/`ffn_out`)をドナーの`gate_proj`/`up_proj`/`down_proj`から初期化し、凍結(`requires_grad=False`)。SSM・埋め込み・LM headのみ学習。
- **Variant B (baseline)**: 完全ランダム初期化、全パラメータ学習。

同じデータ・同じバッチ順序で同じステップ数学習し、lossの推移を比較する。

⚠️ **GPUランタイムを使うこと**(データ準備用ノートブックと違い、今回は実際に学習を回すのでGPUが有効)。

## 1. リポジトリ取得 & 依存ライブラリ

In [ ]:
!git clone https://github.com/fukayatti/BitMC-SSM.git
%cd BitMC-SSM
!pip install -q transformers tiktoken datasets tqdm

import torch
print('CUDA available:', torch.cuda.is_available())

## 2. 動作確認用の小さいデータセットを用意
アーキテクチャの比較が目的なので、英語のTinyStories(既存の`preprocess_data.py`, GPT-2 BPE)で十分。

In [ ]:
!python python/preprocess_data.py --dataset tinystories --num_samples 5000 --out /content/exp_tokens.bin

## 3. ドナーモデル(BitNet-Llama-70M)をロード

In [ ]:
from transformers import AutoModelForCausalLM

DONOR_ID = 'abideen/Bitnet-Llama-70M'
donor = AutoModelForCausalLM.from_pretrained(DONOR_ID)
donor_layers = donor.model.layers
print(f"Donor layers: {len(donor_layers)}")
print(donor.config)

## 4. 実験用モデル定義
`train.py`の`DeltaSSMBlock`(SSM本体, 未改変)を再利用し、FFNの中間次元だけドナーに合わせて設定できる
ブロックを定義する(本番の`BitMCSSMBlock`はFFN中間次元が`d_model*2`に固定なので、ここだけ実験用に別定義)。

In [ ]:
import sys
sys.path.append('python')
import torch.nn as nn
import torch.nn.functional as F

from train import DeltaSSMBlock, FusedRMSNorm
from h_bitlinear import HBitLinear

D_MODEL = 768       # abideen/Bitnet-Llama-70M の hidden_size と一致
FFN_HIDDEN = 1024    # abideen/Bitnet-Llama-70M の intermediate_size と一致
N_LAYERS = 6         # ドナーの層数と一致(全層に1:1でFFNを移植するため)
D_STATE = 32
VOCAB_SIZE = 50257   # GPT-2 BPE (preprocess_data.py のデフォルトと一致)

class ExpBlock(nn.Module):
    """train.py の BitMCSSMBlock と同じ構造だが、FFN中間次元(ffn_hidden)を
    ドナーモデルに合わせて指定できるようにしたもの。SSM部分は完全に同じ実装。"""
    def __init__(self, d_model, d_state, tau, ffn_hidden):
        super().__init__()
        self.norm1 = FusedRMSNorm(d_model)
        self.ssm = DeltaSSMBlock(d_model=d_model, d_state=d_state, tau=tau)
        self.norm2 = FusedRMSNorm(d_model)
        self.ffn_in = HBitLinear(d_model, ffn_hidden * 2, tau=tau, use_hadamard=False)
        self.ffn_out = HBitLinear(ffn_hidden, d_model, tau=tau, use_hadamard=True)

    def forward(self, x, cached_state=None):
        ssm_out, next_state = self.ssm(self.norm1(x), cached_state)
        x = x + ssm_out
        ffn_p = self.ffn_in(self.norm2(x))
        f1, f2 = ffn_p.chunk(2, dim=-1)
        gated = F.silu(f1) * f2
        x = x + self.ffn_out(gated)
        return x, next_state

class ExpModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, d_state, tau=0.85, ffn_hidden=1536):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            ExpBlock(d_model, d_state, tau, ffn_hidden) for _ in range(n_layers)
        ])
        self.final_norm = FusedRMSNorm(d_model)
        self.lm_head = HBitLinear(d_model, vocab_size, tau=tau, use_hadamard=False)

    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for block in self.blocks:
            x, _ = block(x)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        if targets is None:
            return logits
        return F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

## 5. Variant A (FFN移植・凍結) と Variant B (完全ランダム) を構築
同じseedで初期化してからFFN部分だけ上書きすることで、SSM・埋め込み等の初期値は2モデルで完全に揃える
(違いが本当にFFN移植の有無だけになるように)。

In [ ]:
import copy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(42)
model_b_baseline = ExpModel(VOCAB_SIZE, D_MODEL, N_LAYERS, D_STATE, ffn_hidden=FFN_HIDDEN).to(device)

# Variant A はランダム初期化の baseline を土台にコピーし、FFN部分だけドナーの重みで上書きする
model_a_transplant = copy.deepcopy(model_b_baseline).to(device)

with torch.no_grad():
    for i in range(N_LAYERS):
        donor_mlp = donor_layers[i].mlp
        gate_w = donor_mlp.gate_proj.weight.data.to(device)   # (1024, 768)
        up_w = donor_mlp.up_proj.weight.data.to(device)       # (1024, 768)
        down_w = donor_mlp.down_proj.weight.data.to(device)   # (768, 1024)

        ffn_in_w = torch.cat([gate_w, up_w], dim=0)  # (2048, 768) == ffn_in.weight shape
        assert ffn_in_w.shape == model_a_transplant.blocks[i].ffn_in.weight.shape
        assert down_w.shape == model_a_transplant.blocks[i].ffn_out.weight.shape

        model_a_transplant.blocks[i].ffn_in.weight.copy_(ffn_in_w)
        model_a_transplant.blocks[i].ffn_out.weight.copy_(down_w)
        model_a_transplant.blocks[i].ffn_in.weight.requires_grad = False
        model_a_transplant.blocks[i].ffn_out.weight.requires_grad = False

print("✅ Variant A: FFN transplanted + frozen for all", N_LAYERS, "layers")
print("✅ Variant B: fully random init, fully trainable")
for name, m in [('A (transplant)', model_a_transplant), ('B (baseline)', model_b_baseline)]:
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    total = sum(p.numel() for p in m.parameters())
    print(f"  {name}: {trainable:,} / {total:,} trainable params")

## 6. 同一データ・同一バッチ順序で学習し、lossを比較

In [ ]:
import numpy as np

SEQ_LEN = 128
BATCH_SIZE = 16
N_STEPS = 300
LR = 3e-4

tokens = np.memmap('/content/exp_tokens.bin', dtype=np.uint16, mode='r')
print(f"Total tokens available: {len(tokens):,}")

# 両モデルで完全に同じバッチ順序を使うため、先にバッチのインデックスを全部生成しておく
rng = np.random.default_rng(123)
max_start = len(tokens) - SEQ_LEN - 1
batch_starts = [rng.integers(0, max_start, size=BATCH_SIZE) for _ in range(N_STEPS)]

def get_batch(step_idx):
    starts = batch_starts[step_idx]
    x = np.stack([tokens[s:s+SEQ_LEN] for s in starts]).astype(np.int64)
    y = np.stack([tokens[s+1:s+SEQ_LEN+1] for s in starts]).astype(np.int64)
    return torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)

def train_variant(model, label):
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=LR
    )
    losses = []
    model.train()
    for step in range(N_STEPS):
        x, y = get_batch(step)
        optimizer.zero_grad()
        loss = model(x, targets=y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if (step + 1) % 50 == 0:
            print(f"[{label}] step {step+1}/{N_STEPS}  loss={loss.item():.4f}")
    return losses

print("="*70)
print("Training Variant B (baseline, fully random)...")
losses_b = train_variant(model_b_baseline, 'B-baseline')

print("="*70)
print("Training Variant A (FFN transplanted + frozen)...")
losses_a = train_variant(model_a_transplant, 'A-transplant')

In [ ]:
import matplotlib.pyplot as plt

def smooth(vals, k=10):
    return np.convolve(vals, np.ones(k)/k, mode='valid')

plt.figure(figsize=(10, 5))
plt.plot(smooth(losses_b), label='B: baseline (random init)')
plt.plot(smooth(losses_a), label='A: FFN transplanted + frozen')
plt.xlabel('Step')
plt.ylabel('Loss (smoothed)')
plt.legend()
plt.title('FFN Transplant vs Random Init — Training Loss')
plt.show()

print(f"Final loss (last 20 steps avg) — baseline: {np.mean(losses_b[-20:]):.4f}, transplant: {np.mean(losses_a[-20:]):.4f}")

## 📊 結論の読み方
- **transplantのlossがbaselineより明確に低い/速く下がる** → FFN移植に効果あり。本番300M学習でも採用を検討する価値がある。
- **ほぼ差がない、またはtransplantの方が悪い** → 三値量子化の較正ミスマッチやAttention/SSMの分布差が支配的。素直にゼロから学習する方針(蒸留の活用)に絞る。
- この実験は300ステップ・6層・768次元という小規模設定での結果であり、300Mスケールでそのまま同じ傾向になる保証はない点に注意。